# Scrapper and saving of article in a URLs

## Step 1: Imports & Data Folder

In [ ]:
#imports & setup
import re
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from dataclasses import dataclass

DATA_DIR = Path("input data")
DATA_DIR.mkdir(exist_ok=True)

## Step 2: Configuration — Data Structure & Request Headers

In [ ]:
# Data structure & constants (run once)
@dataclass
class SourceDocument:
    source_type: str
    origin: str
    raw_text: str


HEADERS = {
    # Some sites block requests with no User-Agent; a normal browser UA avoids that.
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

# Tags that are almost never part of the actual article content.
NOISE_TAGS = ["script", "style", "nav", "header", "footer", "aside", "form", "noscript"]

## Step 3: Core Functions — Scrape & Save

In [ ]:
# Functions (run once)
def _slugify(url: str) -> str:
    """Turns a URL into a safe filename, e.g. mindfulambition-net-beginners-mind.txt"""
    slug = re.sub(r"^https?://", "", url)
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", slug).strip("-").lower()
    return slug[:100]


def save_to_file(doc: SourceDocument, folder: Path = DATA_DIR) -> Path:
    """Saves a SourceDocument's raw_text to 'input data/<slug>.txt' and returns the path."""
    filename = f"{_slugify(doc.origin)}.txt"
    out_path = folder / filename
    out_path.write_text(doc.raw_text, encoding="utf-8")
    return out_path


def load_from_url(url: str, timeout: int = 15) -> SourceDocument:
    """Downloads the page at `url` and extracts readable article text."""
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(NOISE_TAGS):
        tag.decompose()

    container = (
        soup.find("article")
        or soup.find("div", class_=lambda c: c and "entry-content" in c)
        or soup.find("div", class_=lambda c: c and "post-content" in c)
        or soup.find("main")
    )
    paragraphs = container.find_all("p") if container else soup.find_all("p")

    text = "\n\n".join(
        p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
    )

    if not text:
        raise ValueError(f"No article text found at {url} — page structure may need a custom selector.")

    return SourceDocument(source_type="url", origin=url, raw_text=text)

## Step 4: Run — Scrape One Article

In [ ]:
# Run the scrapper per URLs
doc = load_from_url("https://mindfulambition.net/beginners-mind/")
saved_path = save_to_file(doc)

print(f"Source: {doc.origin}")
print(f"Extracted {len(doc.raw_text)} characters, {len(doc.raw_text.split())} words")
print(f"Saved to: {saved_path}")
print("---")
print(doc.raw_text[:300], "...")

Source: https://mindfulambition.net/beginners-mind/
Extracted 4251 characters, 715 words
Saved to: input data/mindfulambition-net-beginners-mind.txt
---
I used to commute to downtown Chicago by train every day.

At first, I enjoyed this new facet of life. Thenoveltyof the experience made it fun and exciting.

But that fresh feeling quickly washed away, turning my commute into a repetitive inconvenience.All this extra time spent getting from one plac ...


### 5. Run — Scrape All text Sources

In [10]:
urls = [
    "https://mindfulambition.net/beginners-mind/",
    "https://www.infoprolearning.com/blog/understanding-the-learning-curve-why-its-important-in-employee-training-and-development/",
    "https://iopn.library.illinois.edu/pressbooks/instructioninlibraries/chapter/learning-theories-understanding-how-people-learn/",
    "https://www.cmu.edu/teaching/designteach/design/instructionalstrategies/groupprojects/benefits.html",
]

for url in urls:
    doc = load_from_url(url)
    path = save_to_file(doc)
    print(f"✓ {url} -> {path} ({len(doc.raw_text.split())} words)")

✓ https://mindfulambition.net/beginners-mind/ -> input data/mindfulambition-net-beginners-mind.txt (715 words)
✓ https://www.infoprolearning.com/blog/understanding-the-learning-curve-why-its-important-in-employee-training-and-development/ -> input data/www-infoprolearning-com-blog-understanding-the-learning-curve-why-its-important-in-employee-training.txt (845 words)
✓ https://iopn.library.illinois.edu/pressbooks/instructioninlibraries/chapter/learning-theories-understanding-how-people-learn/ -> input data/iopn-library-illinois-edu-pressbooks-instructioninlibraries-chapter-learning-theories-understanding-.txt (8261 words)
✓ https://www.cmu.edu/teaching/designteach/design/instructionalstrategies/groupprojects/benefits.html -> input data/www-cmu-edu-teaching-designteach-design-instructionalstrategies-groupprojects-benefits-html.txt (414 words)
